# Puuliikide klassifitseerimine: Sentinel-2 lõiked + metsaregister -> hierarhiline Random Forest

**Klassid (7):** `MA, KS, KU, LV, LM, HB, MUU` (kõik muud peapuuliigid -> `MUU`).

**Andmejaotus**
- `pohja_eesti` + `tartumaa` (split = train): mudeli treenimine ja hold-out test
- `louna_eesti` (split = test): sõltumatu valideerimine

**Lähenemine** järgib LiisL/TreeSpecies repo mudelit EXP032:
- sama 14 kihti: 10 riba (B02-B12) + NDVI, NDRE, NDMI, NBR; eraldise keskmine pilvemaskeeritud pikslitest;
- **4-etapiline hierarhiline RF**: (1) okaspuu / lehtpuu / MUU -> (2a) KU vs MA -> (2b-i) LV vs ülejäänud lehtpuud -> (2b-ii) HB vs KS vs LM. Etapid treenitakse tegelikel rühmadel, ennustamisel marsruudib iga etapp eelmise etapi **ennustuse** järgi;
- RF seaded: `n_estimators=400, min_samples_leaf=2, class_weight="balanced", random_state=42`, mediaanimputeerimine;
- võrdluseks treenitakse ka tavaline (flat) RF.

**Erinevused repost** (vt. seaded): repo kasutab 2025. aasta 10-päeva aknaid, kõrgusraster-tunnuseid ja käsitsi tehtud 18 liigisignatuuri-tunnust; siin on aknad valikuline (`FEATURE_MODE`), kõrgus valikuline (`HEIGHT_RASTER`) ja signatuuritunnuseid ei ole.

Sõltuvused: `pip install rasterio geopandas numpy pandas scikit-learn matplotlib joblib`

## Seaded

In [ ]:
import hashlib
import json
import re
import warnings
from pathlib import Path

import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio import features as rfeatures
from rasterio.warp import Resampling, reproject
from rasterio.windows import from_bounds
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score,
                             classification_report, cohen_kappa_score, f1_score)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline

# --- kataloogid (töötab nii juurkaustast kui scripts/ kaustast) ---
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
STANDS_DIR = ROOT / "data/metsaregister"
CLIPS_DIR = ROOT / "data/sentinel2_clips"
FEATURES_DIR = ROOT / "data/features"
MODEL_DIR = ROOT / "data/models"
RESULTS_DIR = ROOT / "data/results"

# --- andmed ---
SRC_CRS = "EPSG:3301"
SPECIES_COL = "peapuuliik_kood"                      # peapuuliigi kood eraldise atribuutides
CLASSES = ["MA", "KS", "KU", "LV", "LM", "HB", "MUU"]

# --- pildid / tunnused ---
YEARS = None                 # nt [2025] -> ainult 2025. a stseenid (repo kasutab ainult 2025); None = kõik aastad
FEATURE_MODE = "seasons"     # "seasons" = 3 sesooni + kõik; "windows10" = 10-päeva aknad (nagu repo)
STATS = ("mean",)            # repo: ainult keskmine; lisa "std" kui soovid
HEIGHT_RASTER = None         # nt ROOT / "data/raw/H_20_21_22_23.tif" (kroonikõrgus, EPSG:3301); None = ainult Sentinel-2
INNER_BUFFER_M = 10          # eraldise serv sisse (segapikslid välja); 0 = ei puhvristata
MIN_STAND_PIXELS = 5         # väiksemad eraldised (peale puhvrit) jäetakse välja
MIN_PIXELS_PER_STAT = 3      # min kehtivaid piksleid tunnuse arvutamiseks
VALID_SCL = (4, 5)           # SCL: taimestik, paljas maa (repo lisab ka 6 = vesi)
MIN_SCENE_VALID = 0.3        # stseen jäetakse välja, kui eraldiste pikslitest on kehtivaid alla selle
MIN_FEATURE_COVERAGE = 0.8   # tunnus jääb, kui igas alas on see olemas >=80 % eraldistest

SEASONS = {                  # aastapäeva (DOY) aknad
    "kevad": (120, 159),     # ~1. mai - 8. juuni
    "suvi_1": (160, 190),    # ~9. juuni - 9. juuli
    "suvi_2": (191, 260),    # ~10. juuli - 17. sept
    "koik": (1, 366),
}


def make_windows(days=10):
    return {f"d{s:03d}": (s, min(s + days - 1, 366)) for s in range(1, 367, days)}


PERIODS = make_windows(10) if FEATURE_MODE == "windows10" else SEASONS

# --- mudel ---
SPLIT_MODE = "spatial"       # "spatial" = ruumiliste plokkide kaupa (ausam), "random" = stratifitseeritud juhuslik
BLOCK_M = 2000               # ploki suurus (m) ruumilise jaotuse jaoks (repo: 5000)
TEST_SIZE = 0.3
RANDOM_STATE = 42
RF_KWARGS = dict(n_estimators=400, max_depth=None, min_samples_leaf=2,
                 class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
# Oversampling (sihtarvud klassi kohta). Repo kasutas {"HB":1200,"LM":1200,"KU":1600,"LV":2000,"MUU":500},
# aga nende treeningandmed on palju suuremad. Sinu andmetel (HB ~20 eraldist) oleks see 50x dubleerimine,
# seepärast on vaikimisi välja lülitatud. Skaleeri sihtarvud oma andmete mahule.
OS_STAGE1 = {}               # nt {"MUU": 100}
OS_SPECIES = {}              # nt {"HB": 80, "LM": 80, "LV": 150}
REBUILD_FEATURES = False     # True = ignoreeri vahemällu salvestatud tunnuseid

for d in (FEATURES_DIR, MODEL_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 1. Metsaregistri andmete kontroll

In [ ]:
summary = json.loads((STANDS_DIR / "summary.json").read_text(encoding="utf-8"))
area_split = {a["name"]: a["split"] for a in summary["areas"]}
print("Alad:", area_split)

for area, split in area_split.items():
    g = gpd.read_file(STANDS_DIR / f"{area}.geojson")
    if SPECIES_COL not in g.columns:
        raise KeyError(f"Veergu {SPECIES_COL!r} pole failis {area}.geojson. Olemasolevad: {list(g.columns)}")
    code_ = g[SPECIES_COL].astype(str).str.strip().str.upper()
    label = np.where(code_.isin(CLASSES[:-1]), code_, "MUU")
    print(f"\n{area} ({split}): {len(g)} eraldist, CRS={g.crs}")
    print("  7 klassi:", pd.Series(label).value_counts().reindex(CLASSES, fill_value=0).to_dict())
    print("  MUU sisse läinud koodid:", code_[label == "MUU"].value_counts().to_dict())

## 2. Tunnuste arvutamine eraldiste kaupa

In [ ]:
NAME_RE = re.compile(r"^S2[ABC]_MSIL2A_(\d{8})T\d{6}_(N\d{4})_R\d{3}_(T\w{5})_")
BAND_NAMES = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"]
INDEX_NAMES = ["NDVI", "NDMI", "NDRE", "NBR"]
LAYER_NAMES = BAND_NAMES + INDEX_NAMES


def get_scenes(area):
    rows = []
    for p in sorted((CLIPS_DIR / area).glob("*.tif")):
        m = NAME_RE.match(p.stem)
        if not m:
            continue
        date = pd.to_datetime(m.group(1), format="%Y%m%d")
        rows.append(dict(path=p, product=p.stem, date=date.date(), year=date.year, doy=date.dayofyear,
                         tile=m.group(3), baseline=m.group(2)))
    sc = pd.DataFrame(rows)
    if YEARS and not sc.empty:
        sc = sc[sc.year.isin(YEARS)].reset_index(drop=True)
    return sc


def read_scene(path, ref=None):
    """Loe lõige {riba: massiiv}; vajadusel joonda viiteruudustikule (ref)."""
    with rasterio.open(path) as src:
        names = list(src.descriptions)
        arr = src.read()
        if ref is None:
            ref = dict(crs=src.crs, transform=src.transform, shape=(src.height, src.width))
        same = (src.crs == ref["crs"] and src.transform == ref["transform"]
                and (src.height, src.width) == ref["shape"])
        if not same:
            out = np.zeros((arr.shape[0],) + ref["shape"], dtype=arr.dtype)
            for i in range(arr.shape[0]):
                reproject(arr[i], out[i], src_transform=src.transform, src_crs=src.crs,
                          dst_transform=ref["transform"], dst_crs=ref["crs"],
                          resampling=Resampling.nearest)
            arr = out
    return dict(zip(names, arr)), ref


def norm_diff(a, b):
    with np.errstate(divide="ignore", invalid="ignore"):
        return ((a - b) / (a + b)).astype("float32")


def scene_layers(bands, labels, baseline):
    """Peegelduvused + indeksid, pilvemaskiga. Tagasta (kihid | None, kehtivate osakaal)."""
    ok = np.isin(bands["SCL"], VALID_SCL) & (bands["B02"] > 0)
    inside = labels > 0
    valid_frac = float(ok[inside].mean()) if inside.any() else 0.0
    if valid_frac < MIN_SCENE_VALID:
        return None, valid_frac
    offset = 1000 if int(baseline[1:]) >= 400 else 0       # BOA_ADD_OFFSET (baseline >= N0400)
    refl = {b: np.where(ok, np.clip((bands[b].astype("float32") - offset) / 10000, 0, None), np.nan)
                 .astype("float32") for b in BAND_NAMES}
    layers = dict(refl)
    layers["NDVI"] = norm_diff(refl["B08"], refl["B04"])
    layers["NDMI"] = norm_diff(refl["B08"], refl["B11"])
    layers["NDRE"] = norm_diff(refl["B8A"], refl["B05"])
    layers["NBR"] = norm_diff(refl["B08"], refl["B12"])
    return layers, valid_frac


def zonal_stats(labels, n, arr):
    """Eraldise keskmine ja std (labels: 0 = taust, 1..n = eraldis)."""
    m = (labels > 0) & np.isfinite(arr)
    lab, v = labels[m], arr[m].astype("float64")
    cnt = np.bincount(lab, minlength=n + 1)
    s = np.bincount(lab, weights=v, minlength=n + 1)
    s2 = np.bincount(lab, weights=v * v, minlength=n + 1)
    with np.errstate(divide="ignore", invalid="ignore"):
        mean = s / cnt
        std = np.sqrt(np.maximum(s2 / cnt - mean ** 2, 0))
    bad = cnt < MIN_PIXELS_PER_STAT
    mean[bad] = np.nan
    std[bad] = np.nan
    return mean[1:], std[1:]


def height_features(gdf, raster_path):
    """Kõrgusrasteri zonaalstatistika (mean, median, std, min, max) eraldiste kaupa."""
    with rasterio.open(raster_path) as src:
        g = gdf.to_crs(src.crs)
        win = from_bounds(*g.total_bounds, transform=src.transform).round_offsets().round_lengths()
        arr = src.read(1, window=win, masked=True).astype("float32").filled(np.nan)
        transform = src.window_transform(win)
    shapes = [(geom, i + 1) for i, geom in enumerate(g.geometry) if geom is not None and not geom.is_empty]
    lab = rfeatures.rasterize(shapes, out_shape=arr.shape, transform=transform, fill=0, dtype="int32")
    m = (lab > 0) & np.isfinite(arr)
    d = pd.DataFrame({"i": lab[m] - 1, "h": arr[m]})
    st = d.groupby("i")["h"].agg(["mean", "median", "std", "min", "max"]).add_prefix("height_")
    return st.reindex(range(len(gdf)))


def settings_hash(area):
    sc = get_scenes(area)
    key = json.dumps(dict(periods=PERIODS, years=YEARS, stats=STATS, buf=INNER_BUFFER_M, scl=VALID_SCL,
                          msv=MIN_SCENE_VALID, mpp=MIN_PIXELS_PER_STAT, height=str(HEIGHT_RASTER),
                          scenes=sc["product"].tolist() if not sc.empty else []),
                     sort_keys=True, default=str)
    return hashlib.md5(key.encode()).hexdigest()[:8]


def area_features(area, split):
    gdf = gpd.read_file(STANDS_DIR / f"{area}.geojson")
    gdf = gdf.set_crs(SRC_CRS) if gdf.crs is None else gdf.to_crs(SRC_CRS)
    gdf = gdf.reset_index(drop=True)
    n = len(gdf)

    scenes = get_scenes(area)
    if scenes.empty:
        raise FileNotFoundError(f"{CLIPS_DIR / area}: sobivaid lõikeid ei leitud (YEARS={YEARS}). "
                                "Käivita enne lõikamise märkmik.")

    # viiteruudustik esimesest stseenist, eraldiste rasteriseerimine sellele
    _, ref = read_scene(scenes.path.iloc[0])
    geoms = gdf.geometry.buffer(-INNER_BUFFER_M) if INNER_BUFFER_M else gdf.geometry
    geoms = gpd.GeoSeries(geoms, crs=SRC_CRS).to_crs(ref["crs"])
    shapes = [(g, i + 1) for i, g in enumerate(geoms) if g is not None and not g.is_empty]
    labels = rfeatures.rasterize(shapes, out_shape=ref["shape"], transform=ref["transform"],
                                 fill=0, dtype="int32")
    n_pix = np.bincount(labels.ravel(), minlength=n + 1)[1:]

    used, log = [], []
    for r in scenes.itertuples():
        bands, ref = read_scene(r.path, ref)
        layers, vf = scene_layers(bands, labels, r.baseline)
        log.append(dict(product=r.product, date=r.date, tile=r.tile, valid=round(vf, 3),
                        used=layers is not None))
        if layers is not None:
            used.append((r.doy, layers))
    print(f"\n{area}: stseenid\n", pd.DataFrame(log).to_string(index=False))

    feats = {}
    for period, (d0, d1) in PERIODS.items():
        sel = [l for doy, l in used if d0 <= doy <= d1]
        if not sel:
            continue
        for lname in LAYER_NAMES:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                comp = np.nanmedian(np.stack([l[lname] for l in sel]), axis=0)
            mean, std = zonal_stats(labels, n, comp)
            if "mean" in STATS:
                feats[f"{period}__{lname}__mean"] = mean
            if "std" in STATS:
                feats[f"{period}__{lname}__std"] = std

    df = pd.DataFrame(feats)
    if HEIGHT_RASTER:
        h = height_features(gdf, HEIGHT_RASTER)
        for c in h.columns:
            df[c] = h[c].values
    code_ = gdf[SPECIES_COL].astype(str).str.strip().str.upper()
    cen = gdf.geometry.centroid
    df.insert(0, "stand_idx", np.arange(n))
    df.insert(1, "area", area)
    df.insert(2, "split", split)
    df.insert(3, "species_raw", code_.values)
    df.insert(4, "label", np.where(code_.isin(CLASSES[:-1]), code_, "MUU"))
    df.insert(5, "n_pixels", n_pix)
    df.insert(6, "x", cen.x.values)
    df.insert(7, "y", cen.y.values)
    return df[df.n_pixels >= MIN_STAND_PIXELS].reset_index(drop=True)


frames = []
for area, split in area_split.items():
    cache = FEATURES_DIR / f"{area}_features_{settings_hash(area)}.csv"
    if cache.exists() and not REBUILD_FEATURES:
        df_a = pd.read_csv(cache)
        print(f"\n{area}: tunnused vahemälust ({len(df_a)} eraldist)")
    else:
        df_a = area_features(area, split)
        df_a.to_csv(cache, index=False)
    frames.append(df_a)

df = pd.concat(frames, ignore_index=True)
print(f"\nKokku {len(df)} eraldist")
display(df.groupby(["area", "label"]).size().unstack(fill_value=0).reindex(columns=CLASSES, fill_value=0))

## 3. Tunnuste valik (ühised kõigis alades)

In [ ]:
META_COLS = ["stand_idx", "area", "split", "species_raw", "label", "n_pixels", "x", "y"]
candidate = [c for c in df.columns if c not in META_COLS]

# tunnus jääb, kui see on olemas igas alas (ka testalas), et mudel ei näeks puuduvaid veerge
coverage = df.groupby("area")[candidate].agg(lambda s: s.notna().mean())
FEATURES = [c for c in candidate if coverage[c].min() >= MIN_FEATURE_COVERAGE]
periods_kept = sorted({c.split("__")[0] for c in FEATURES if "__" in c})
print(f"Tunnuseid: {len(FEATURES)} / {len(candidate)}; perioodid kõigis alades: {periods_kept}")

periods_all = sorted({c.split("__")[0] for c in candidate if "__" in c})
period_cov = pd.DataFrame({p: coverage[[c for c in candidate if c.startswith(p + "__")]].mean(axis=1)
                           for p in periods_all})
print("\nPerioodide kaetus alade kaupa (osa eraldistest, millel tunnused olemas):")
display(period_cov.round(2))
if not FEATURES:
    raise RuntimeError("Ühiseid tunnuseid pole: alade stseenid ei kata samu perioode. "
                       "Kasuta FEATURE_MODE = 'seasons' või lae alla tihedam ühine aegrida.")

## 4. Mudelid: hierarhiline RF (EXP032 stiilis) ja tavaline RF

Etapid: `group` (CONIFER / DECIDUOUS / MUU) -> `conifer` (KU vs MA) -> `lv` (LV vs REST) -> `rest` (HB vs KS vs LM).
Etapid treenitakse **tegelikel** rühmadel; ennustamisel marsruudib iga etapp eelmise etapi **ennustuse** järgi (testsildid ei leki).

In [ ]:
GROUP_MAP = {"KU": "CONIFER", "MA": "CONIFER",
             "HB": "DECIDUOUS", "KS": "DECIDUOUS", "LM": "DECIDUOUS", "LV": "DECIDUOUS",
             "MUU": "MUU"}
CONIFER = {"KU", "MA"}
DECIDUOUS = {"HB", "KS", "LM", "LV"}
REST_DECIDUOUS = {"HB", "KS", "LM"}


def make_model():
    return Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("rf", RandomForestClassifier(**RF_KWARGS))])


def oversample(X, y, strategy, seed=RANDOM_STATE):
    """Juhuslik oversampling asendusega; strategy = {klass: sihtarv} (ainult kui klass on sihtarvust väiksem)."""
    if not strategy:
        return X, y
    rng = np.random.default_rng(seed)
    yv = np.asarray(y)
    idx = list(range(len(yv)))
    for cls, target in strategy.items():
        pos = np.where(yv == cls)[0]
        if 0 < len(pos) < target:
            idx.extend(rng.choice(pos, target - len(pos), replace=True))
    idx = np.array(idx)
    return X.iloc[idx].reset_index(drop=True), pd.Series(yv[idx])


class Stage:
    """Üks hierarhia etapp: RF, või konstantne ennustaja kui treeningus on ainult üks klass."""

    def fit(self, X, y, os_strategy=None):
        X = X.reset_index(drop=True)
        y = pd.Series(np.asarray(y))
        X, y = oversample(X, y, os_strategy)
        self.classes_ = sorted(y.unique())
        self.model, self.const = None, (self.classes_[0] if len(self.classes_) == 1 else None)
        if len(self.classes_) >= 2:
            self.model = make_model().fit(X, y)
        return self

    def predict(self, X):
        if self.model is None:
            return np.full(len(X), self.const, dtype=object)
        return self.model.predict(X)

    def proba(self, X):
        if self.model is None:
            return pd.DataFrame({self.const: np.ones(len(X))}) if self.const is not None else pd.DataFrame(index=range(len(X)))
        return pd.DataFrame(self.model.predict_proba(X), columns=self.model.classes_)


def train_hier(X, y):
    X = X.reset_index(drop=True)
    y = pd.Series(np.asarray(y))
    m = {"group": Stage().fit(X, y.map(GROUP_MAP), OS_STAGE1)}
    con = y.isin(CONIFER)
    m["conifer"] = Stage().fit(X[con], y[con], OS_SPECIES)
    dec = y.isin(DECIDUOUS)
    y_lv = y[dec].where(y[dec] == "LV", "REST")
    m["lv"] = Stage().fit(X[dec], y_lv, {"LV": OS_SPECIES["LV"]} if "LV" in OS_SPECIES else None)
    rest = y.isin(REST_DECIDUOUS)
    m["rest"] = Stage().fit(X[rest], y[rest], OS_SPECIES)
    return m


def predict_hier(X, m):
    """Realistlik ennustus: iga etapp marsruudib eelmise etapi ENNUSTUSE järgi. Tagasta (liik, rühm)."""
    X = X.reset_index(drop=True)
    grp = m["group"].predict(X)
    out = np.full(len(X), "MUU", dtype=object)
    con = grp == "CONIFER"
    if con.any():
        out[con] = m["conifer"].predict(X[con])
    dec = grp == "DECIDUOUS"
    if dec.any():
        idx = np.where(dec)[0]
        is_lv = m["lv"].predict(X.iloc[idx]) == "LV"
        out[idx[is_lv]] = "LV"
        ridx = idx[~is_lv]
        if len(ridx):
            out[ridx] = m["rest"].predict(X.iloc[ridx])
    return out, grp


def predict_proba_hier(X, m):
    """Klasside tõenäosused etappide tõenäosuste korrutisena (ainult raporteerimiseks)."""
    X = X.reset_index(drop=True)
    n = len(X)
    P = pd.DataFrame(0.0, index=range(n), columns=CLASSES)
    get = lambda df, c: df[c].values if c in df.columns else np.zeros(n)
    g, c, lv, r = (m[k].proba(X) for k in ("group", "conifer", "lv", "rest"))
    P["MUU"] = get(g, "MUU")
    for s in ("KU", "MA"):
        P[s] = get(g, "CONIFER") * get(c, s)
    P["LV"] = get(g, "DECIDUOUS") * get(lv, "LV")
    for s in ("HB", "KS", "LM"):
        P[s] = get(g, "DECIDUOUS") * get(lv, "REST") * get(r, s)
    return P


def evaluate(title, y_true, y_pred):
    labs = sorted(set(y_true))
    m = dict(accuracy=accuracy_score(y_true, y_pred),
             balanced_accuracy=balanced_accuracy_score(y_true, y_pred),
             f1_macro=f1_score(y_true, y_pred, labels=labs, average="macro", zero_division=0),
             kappa=cohen_kappa_score(y_true, y_pred))
    print(f"\n=== {title} ===")
    print({k: round(float(v), 3) for k, v in m.items()})
    print(classification_report(y_true, y_pred, labels=sorted(set(y_true) | set(y_pred)), zero_division=0))
    return {k: float(v) for k, v in m.items()}

## 5. Hold-out test (pohja_eesti + tartumaa)

In [ ]:
train_areas = [a for a, s in area_split.items() if s == "train"]
val_areas = [a for a, s in area_split.items() if s == "test"]
tr = df[df.area.isin(train_areas)].reset_index(drop=True)
va = df[df.area.isin(val_areas)].reset_index(drop=True)
print("Treening:", train_areas, len(tr), "| Valideerimine:", val_areas, len(va))

if SPLIT_MODE == "spatial":
    blocks = (tr.area + "_" + (tr.x // BLOCK_M).astype(int).astype(str)
              + "_" + (tr.y // BLOCK_M).astype(int).astype(str))
    idx_fit, idx_hold = next(GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE,
                                               random_state=RANDOM_STATE).split(tr, groups=blocks))
else:
    idx_fit, idx_hold = train_test_split(np.arange(len(tr)), test_size=TEST_SIZE,
                                         stratify=tr.label, random_state=RANDOM_STATE)

X_fit, y_fit = tr.loc[idx_fit, FEATURES], tr.loc[idx_fit, "label"]
X_hold, y_hold = tr.loc[idx_hold, FEATURES], tr.loc[idx_hold, "label"]

flat_h = make_model().fit(X_fit, y_fit)
hier_h = train_hier(X_fit, y_fit)
pred_hold_flat = flat_h.predict(X_hold)
pred_hold_hier, grp_hold = predict_hier(X_hold, hier_h)

title = f"Hold-out ({SPLIT_MODE}, {len(idx_hold)} eraldist)"
metrics_holdout_flat = evaluate(f"{title}: FLAT RF", y_hold, pred_hold_flat)
metrics_holdout_hier = evaluate(f"{title}: HIERARHILINE RF", y_hold, pred_hold_hier)

## 6. Lõplikud mudelid ja valideerimine (louna_eesti)

In [ ]:
X_tr, y_tr = tr[FEATURES], tr["label"]          # kogu treeningandmed (hold-out tagasi)
flat = make_model().fit(X_tr, y_tr)
hier = train_hier(X_tr, y_tr)

X_va, y_va = va[FEATURES], va["label"]
pred_val_flat = flat.predict(X_va)
pred_val_hier, grp_val = predict_hier(X_va, hier)
proba_val = predict_proba_hier(X_va, hier)

title = f"Valideerimine {val_areas} ({len(va)} eraldist)"
metrics_val_flat = evaluate(f"{title}: FLAT RF", y_va, pred_val_flat)
metrics_val_hier = evaluate(f"{title}: HIERARHILINE RF", y_va, pred_val_hier)
metrics_val_group = evaluate("Etapp 1 (okaspuu/lehtpuu/MUU), valideerimine", y_va.map(GROUP_MAP), grp_val)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, pred, name in ((axes[0], pred_val_flat, "FLAT RF"), (axes[1], pred_val_hier, "HIERARHILINE RF")):
    ConfusionMatrixDisplay.from_predictions(y_va, pred, labels=CLASSES, normalize="true",
                                            values_format=".2f", cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(f"{val_areas[0]}: {name}")
plt.tight_layout()
plt.show()


def importance_series(rf):
    return pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)


def importance_groups(imp):
    period = imp.index.map(lambda c: c.split("__")[0] if "__" in c else "aux")
    layer = imp.index.map(lambda c: c.split("__")[1] if "__" in c else c)
    return imp.groupby(period).sum().round(3), imp.groupby(layer).sum().sort_values(ascending=False).round(3)


imp_flat = importance_series(flat.named_steps["rf"])
imp_flat.head(25)[::-1].plot.barh(figsize=(7, 7), title="Flat RF: top 25 tunnust")
plt.tight_layout()
plt.show()

tbl = {"flat (kõik klassid)": imp_flat}
if hier["group"].model is not None:
    tbl["etapp 1 (rühm)"] = importance_series(hier["group"].model.named_steps["rf"])
for name, imp in tbl.items():
    by_period, by_layer = importance_groups(imp)
    print(f"\n--- Olulisus: {name} ---")
    print("perioodide kaupa:\n", by_period.to_string())
    print("kihtide kaupa:\n", by_layer.to_string())

## 7. Salvestamine

In [ ]:
joblib.dump(dict(hier=hier, flat=flat, features=FEATURES, classes=CLASSES, group_map=GROUP_MAP,
                 settings=dict(periods=PERIODS, feature_mode=FEATURE_MODE, years=YEARS, stats=STATS,
                               inner_buffer_m=INNER_BUFFER_M, valid_scl=VALID_SCL,
                               min_scene_valid=MIN_SCENE_VALID, rf_kwargs=RF_KWARGS)),
            MODEL_DIR / "rf_species.joblib")

(RESULTS_DIR / "metrics.json").write_text(json.dumps(
    dict(holdout_flat=metrics_holdout_flat, holdout_hier=metrics_holdout_hier,
         validation_flat=metrics_val_flat, validation_hier=metrics_val_hier,
         validation_stage1_group=metrics_val_group,
         n_features=len(FEATURES), feature_mode=FEATURE_MODE, split_mode=SPLIT_MODE,
         train_areas=train_areas, val_areas=val_areas),
    indent=2, ensure_ascii=False), encoding="utf-8")

# ennustused valideerimisala eraldistele (EPSG:3301 GeoJSON)
pred = va[["area", "stand_idx"]].copy()
pred["true_class"] = y_va.values
pred["pred_hier"] = pred_val_hier
pred["pred_flat"] = pred_val_flat
pred["group_pred"] = grp_val
pred["p_pred_hier"] = [round(float(proba_val.at[i, c]), 3) if c in proba_val.columns else 0.0
                       for i, c in enumerate(pred_val_hier)]
pred["correct_hier"] = pred["true_class"] == pred["pred_hier"]
pred["correct_flat"] = pred["true_class"] == pred["pred_flat"]
for area in val_areas:
    g = gpd.read_file(STANDS_DIR / f"{area}.geojson").reset_index(drop=True)
    g["stand_idx"] = g.index
    g = g.merge(pred[pred.area == area].drop(columns="area"), on="stand_idx", how="inner")
    out = RESULTS_DIR / f"{area}_predictions.geojson"
    g.to_file(out, driver="GeoJSON")
    print("Salvestatud:", out)
print("Mudel:", MODEL_DIR / "rf_species.joblib")

## 8. Kokkuvõte: põhilised mõõdikud (flat vs hierarhiline)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support


def metrics_summary(y_true, y_pred):
    """Üldmõõdikud + klasside tabel (precision, recall, F1, support)."""
    overall = {"accuracy": accuracy_score(y_true, y_pred),
               "balanced_accuracy (macro recall)": balanced_accuracy_score(y_true, y_pred),
               "kappa": cohen_kappa_score(y_true, y_pred)}
    for avg in ("macro", "weighted"):
        p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=sorted(set(y_true)),
                                                     average=avg, zero_division=0)
        overall.update({f"precision_{avg}": p, f"recall_{avg}": r, f"f1_{avg}": f})
    p, r, f, sup = precision_recall_fscore_support(y_true, y_pred, labels=CLASSES, zero_division=0)
    per_class = pd.DataFrame({"precision": p, "recall": r, "f1": f, "support": sup}, index=CLASSES)
    per_class["predicted"] = [int((np.asarray(y_pred) == c).sum()) for c in CLASSES]
    return pd.Series(overall), per_class


results = {
    "Hold-out FLAT": (y_hold.values, pred_hold_flat),
    "Hold-out HIERARHILINE": (y_hold.values, pred_hold_hier),
    f"Valideerimine FLAT ({', '.join(val_areas)})": (y_va.values, pred_val_flat),
    f"Valideerimine HIERARHILINE ({', '.join(val_areas)})": (y_va.values, pred_val_hier),
}
overall_tbl, per_class_tbls = {}, {}
for name, (yt, yp) in results.items():
    overall_tbl[name], per_class_tbls[name] = metrics_summary(yt, yp)
    print(f"\n{'=' * 60}\n{name}: {len(yt)} eraldist\n{'=' * 60}")
    print(overall_tbl[name].round(3).to_string())
    print()
    display(per_class_tbls[name].round(3))

print("\nÜldmõõdikute võrdlus:")
display(pd.DataFrame(overall_tbl).round(3))

for name, tbl in per_class_tbls.items():
    safe = re.sub(r"\W+", "_", name).strip("_")
    tbl.round(4).to_csv(RESULTS_DIR / f"per_class_{safe}.csv")
pd.DataFrame(overall_tbl).round(4).to_csv(RESULTS_DIR / "overall_metrics.csv")